In [ ]:
from astrapy import DataAPIClient
# Initialize the client 
client = DataAPIClient() 
db = client.get_database( 
  "https://d23e5536-e548-48a9-b764-25e8fcbafd39-us-east-2.apps.astra.datastax.com", 
  token="AstraCS:PLTQMkUzNwpeiMorexqDbrJz:73692a8977a5b42de0b221d55e10c90a1fda728740ad682798c18d2b7f102965",
) 
# db.create_collection("test_collection")
# db.drop_collection("rag_learning")
print(f"Connected to Astra DB. Collections: {db.list_collection_names()}")


Connected to Astra DB. Collections: ['rag_learning', 'test_collection']


In [ ]:
from langchain_astradb import AstraDBVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model = "sentence-transformers/all-MiniLM-L6-v2")
vector_store_db = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint= "https://d23e5536-e548-48a9-b764-25e8fcbafd39-us-east-2.apps.astra.datastax.com",
    token="AstraCS:PLTQMkUzNwpeiMorexqDbrJz:73692a8977a5b42de0b221d55e10c90a1fda728740ad682798c18d2b7f102965",
    collection_name="rag_learning"
)   

In [7]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [12]:
vector_store_db.add_documents(documents=documents)

['06a212d17bbd4c8bbb6556921fb5f50d',
 'fb713a1d4dd94f55a8eeb717ecb64df7',
 'f8a7f20c90e0489fb4a865b396529d93',
 'e34f2663634445beb66c0c4139554779',
 '1f37435485e24db1bfe7c00076ce7d3b',
 '62916dfae8874c8fb9bf6a7bc68f9fd8',
 '057009c5d1d64df5ba4d0029259080aa',
 '9cd392f272b542a48e807bd2b5a617cf',
 'c8cc6883c9aa4cb3a87ed320adf80b6c',
 '13c70a33129846e4acc332622f7f18a5']

In [15]:
result = vector_store_db.similarity_search("Why do we learn lang graph")
result

[Document(id='9cd392f272b542a48e807bd2b5a617cf', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='f8a7f20c90e0489fb4a865b396529d93', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='62916dfae8874c8fb9bf6a7bc68f9fd8', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(id='13c70a33129846e4acc332622f7f18a5', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [21]:
result = vector_store_db.similarity_search_with_score(
    "What is the weather today?", k=2, filter={'source':'tweet'}
)
result

[(Document(id='06a212d17bbd4c8bbb6556921fb5f50d', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
  0.5917739),
 (Document(id='13c70a33129846e4acc332622f7f18a5', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
  0.5353462)]

In [37]:
vector_reteriver = vector_store_db.as_retriever(search_kwargs={"k":3, "score_threshold":0.8},search_type="similarity_score_threshold")
vector_reteriver.invoke("What is weather today")

[Document(id='fb713a1d4dd94f55a8eeb717ecb64df7', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [38]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_ollama import OllamaLLM
llm = OllamaLLM(
    model="hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"
)

system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

context: {context}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
    # ("ai", "Hi there, how can I help you?")
])

document_chain = create_stuff_documents_chain(llm, prompt_template)
rag_chain = create_retrieval_chain(vector_reteriver, document_chain)


In [ ]:
result = rag_chain.invoke({"input":"How is the weather today"})
# result
result['answer']

{'input': 'How is the weather today',
 'context': [Document(id='fb713a1d4dd94f55a8eeb717ecb64df7', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')],
 'answer': "The current weather is mostly cloudy and overcast, with a temperature of around 62°F (16°C). There's no indication of any significant changes in the weather forecast. Overall, it's a relatively ordinary day with partly sunny skies in the afternoon."}

In [45]:
result = rag_chain.invoke({"input":"How much point stock market is down today"})
result

{'input': 'How much point stock market is down today',
 'context': [Document(id='c8cc6883c9aa4cb3a87ed320adf80b6c', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')],
 'answer': "The current decline in the stock market is approximately 500 points, which translates to around 2% of the total market value. I don't know at this time regarding the exact percentage or changes in the overall market value. It's essential to note that past performance does not guarantee future results and that the stock market can fluctuate rapidly."}